# Data Profiling

### Environment Setup

In [7]:
import numpy as np
import pandas as pd

from chicago_food_inspection.data_loader import load_data
from chicago_food_inspection.logging_config import get_logger, setup_logging
from chicago_food_inspection.paths import PROJECT_ROOT

setup_logging()
logger = get_logger("notebooks.01_data_profiling")

logger.info("Notebook 01_data_profiling started.")

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Project root: {PROJECT_ROOT}")

2026-10-16 20:47:01,075 | notebooks.01_data_profiling | INFO | Notebook 01_data_profiling started.
NumPy: 2.5.3
Pandas: 3.0.6
Project root: D:\ML\Portfolio\Projects\chicago-food-inspection-analysis


### Loading Data

In [6]:
df = load_data()

D:\ML\Portfolio\Projects\chicago-food-inspection-analysis\src\chicago_food_inspection\data_loader.py:29: DtypeWarning: Columns (0: Inspection ID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, encoding="utf-8", usecols=usecols, dtype=dtypes)


2026-10-16 20:46:40,776 | chicago_food_inspection.data_loader | INFO | Loaded 314555 rows × 17 columns from data\raw\Food_Inspections_20260914.csv (503.8 MB in memory)


### Basic Info

In [8]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 314555 entries, 0 to 314554
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Inspection ID    314555 non-null  object 
 1   DBA Name         314550 non-null  str    
 2   AKA Name         312153 non-null  str    
 3   License #        314531 non-null  float64
 4   Facility Type    309259 non-null  str    
 5   Risk             314463 non-null  str    
 6   Address          314550 non-null  str    
 7   City             314372 non-null  str    
 8   State            314475 non-null  str    
 9   Zip              314509 non-null  float64
 10  Inspection Date  314550 non-null  str    
 11  Inspection Type  314549 non-null  str    
 12  Results          314550 non-null  str    
 13  Violations       225926 non-null  str    
 14  Latitude         313514 non-null  float64
 15  Longitude        313514 non-null  float64
 16  Location         313514 non-null  str    
dtypes:

In [10]:
df.shape

(314555, 17)

In [11]:
df.dtypes

Inspection ID       object
DBA Name               str
AKA Name               str
License #          float64
Facility Type          str
Risk                   str
Address                str
City                   str
State                  str
Zip                float64
Inspection Date        str
Inspection Type        str
Results                str
Violations             str
Latitude           float64
Longitude          float64
Location               str
dtype: object

In [9]:
df.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
0,2642587,HALAL MANDI HOUSE,HALAL MANDI HOUSE,3073458.0,Restaurant,Risk 1 (High),6240 N CALIFORNIA AVE,CHICAGO,IL,60659.0,09/11/2026,Complaint Re-Inspection,Fail,37. FOOD PROPERLY LABELED; ORIGINAL CONTAINER - Comments: 3-602.11 MUST LAB...,41.995177,-87.699705,"(41.995177101341675, -87.69970469303253)"
1,2642546,BENNY'S PIZZA,BENNY'S PIZZA,2501304.0,Restaurant,Risk 1 (High),1244 W 18TH ST,CHICAGO,IL,60608.0,09/10/2026,Complaint Re-Inspection,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: ...,41.858031,-87.658224,"(41.85803142125194, -87.6582243375243)"
2,2642505,Cha TraMue,Cha TraMue,3100382.0,Restaurant,Risk 2 (Medium),1616 N DAMEN AVE,CHICAGO,IL,60647.0,09/10/2026,License,Fail,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: ...,41.910974,-87.677622,"(41.91097377355574, -87.67762165938252)"
3,2642484,LAZARO CARDENAS SOUTH,LAZARO CARDENAS,3759765.0,School,Risk 1 (High),2524 S Central Park (3600W),CHICAGO,IL,60623.0,09/10/2026,Canvass,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: ...,41.845443,-87.714964,"(41.845442959859945, -87.71496351888159)"
4,2642519,MOZART ELEMENTARY SCHOOL,MOZART ELEM.SCHOOL & BRANCH ANNEX,24611.0,School,Risk 1 (High),2200 N Hamlin AVE,CHICAGO,IL,60647.0,09/10/2026,Canvass,Fail,56. ADEQUATE VENTILATION & LIGHTING; DESIGNATED AREAS USED - Comments: OBSER...,41.920927,-87.721925,"(41.920927342186204, -87.72192541073612)"


### Column Overview

In [13]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "unique": df.nunique(),
})
overview

,dtype,non_null,unique
Inspection ID,object,314555,314555
DBA Name,str,314550,35076
AKA Name,str,312153,33370
License #,float64,314531,48995
Facility Type,str,309259,527
Risk,str,314463,4
Address,str,314550,33252
City,str,314372,94
State,str,314475,7
Zip,float64,314509,139


### Missing Values

In [12]:
summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(2),
})
summary = summary.sort_values("missing_pct", ascending=False)
summary.loc["TOTAL"] = [df.isna().sum().sum(), (df.isna().sum().sum() / df.size * 100).round(2)]
summary.style.format({"missing_count": "{:.0f}", "missing_pct": "{:.2f}%"})

,missing_count,missing_pct
Violations,88629,28.18%
Facility Type,5296,1.68%
AKA Name,2402,0.76%
Longitude,1041,0.33%
Location,1041,0.33%
Latitude,1041,0.33%
City,183,0.06%
State,80,0.03%
Risk,92,0.03%
License #,24,0.01%


### Duplicate Analysis

In [14]:
print(f"Fully duplicated rows: {df.duplicated().sum():,}")
print(f"Duplicate Inspection IDs: {df['Inspection ID'].duplicated().sum():,}")
print(f"Unique facilities (License #): {df['License #'].nunique():,}")

Fully duplicated rows: 0
Duplicate Inspection IDs: 0
Unique facilities (License #): 48,995


### Categorical Columns

In [16]:
categorical_columns = ["Risk", "State", "Results"]
for categorical_column in categorical_columns:
    print(f"\n--- {categorical_column} ---")
    print(df[categorical_column].value_counts())


--- Risk ---
Risk
Risk 1 (High)      234059
Risk 2 (Medium)     55894
Risk 3 (Low)        24425
All                    85
Name: count, dtype: int64

--- State ---
State
IL    314451
IN        13
CA         5
WI         3
DC         1
CO         1
NY         1
Name: count, dtype: int64

--- Results ---
Results
Pass                    162667
Fail                     60512
Pass w/ Conditions       46743
Out of Business          25842
No Entry                 14104
Not Ready                 4587
Business Not Located        95
Name: count, dtype: int64


### Date Analysis

In [17]:
df["Inspection Date"] = pd.to_datetime(df["Inspection Date"], errors="coerce")
print(f"Date range: {df['Inspection Date'].min()} → {df['Inspection Date'].max()}")
print(f"Unparseable dates: {df['Inspection Date'].isna().sum():,}")
print(df["Inspection Date"].dt.year.value_counts().sort_index())

Date range: 2010-01-21 00:00:00 → 2026-09-11 00:00:00
Unparseable dates: 5
Inspection Date
2010.0    17209
2011.0    18747
2012.0    18866
2013.0    20947
2014.0    21540
2015.0    20911
2016.0    22818
2017.0    21587
2018.0    17192
2019.0    19052
2020.0    15123
2021.0    15868
2022.0    16936
2023.0    18261
2024.0    19228
2025.0    19205
2026.0    11060
Name: count, dtype: int64


### Numeric Column Statistics

In [18]:
df.describe(include="number")

,License #,Zip,Latitude,Longitude
count,3.145310e+05,314509.000000,313514.000000,313514.000000
mean,1.806646e+06,60628.694664,41.880597,-87.676388
std,9.480431e+05,193.662236,0.080853,0.058461
min,0.000000e+00,10014.000000,41.644670,-87.906874
25%,1.473123e+06,60614.000000,41.832134,-87.707384
50%,2.103379e+06,60625.000000,41.891733,-87.666350
75%,2.458745e+06,60643.000000,41.939486,-87.634868
max,9.999999e+06,91706.000000,42.021064,-87.525094
